<a href="https://colab.research.google.com/github/IrisCheon/NLP-practice/blob/main/Topic_Modeling(LDA).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LDA Topic Modeling on 20 Newsgroups

## Project Overview

This project explores unsupervised topic modeling using Latent Dirichlet Allocation (LDA) on a subset of the 20 Newsgroups dataset.

Unlike previous NLP projects based on supervised classification, this project focuses on discovering latent topic structures without using target labels during training. Documents were converted into document-term matrices using CountVectorizer, and LDA was used to identify recurring word co-occurrence patterns across the corpus.

The project also compares unigram and bigram representations to examine how text representation affects topic coherence and topic separability.

## Main Objectives

- Understand the basic workflow of LDA topic modeling
- Explore document-topic and topic-word distributions
- Interpret topics using representative words and documents
- Compare unigram and bigram feature representations
- Analyze topic overlap and topic separation in unsupervised NLP

## Dataset

- Dataset: 20 Newsgroups
- Categories used:
  - `comp.graphics`
  - `sci.space`
  - `rec.sport.baseball`
  - `alt.atheism`
  - `talk.religion.misc`

## Methods

- `CountVectorizer`
- `LatentDirichletAllocation (LDA)`
- Topic-word analysis
- Document-topic distribution analysis
- Crosstab comparison between dominant topics and original labels

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [4]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# ■ 데이터 확인

In [18]:
type(dataset)

sklearn.utils._bunch.Bunch

In [19]:
print(dataset.keys())

dict_keys(['data', 'filenames', 'target_names', 'target', 'DESCR'])


In [26]:
dataset.data[0]

'I was wondering if anyone out there could enlighten me on this car I saw\nthe other day. It was a 2-door sports car, looked to be from the late 60s/\nearly 70s. It was called a Bricklin. The doors were really small. In addition,\nthe front bumper was separate from the rest of the body. This is \nall I know. If anyone can tellme a model name, engine specs, years\nof production, where this car is made, history, or whatever info you\nhave on this funky looking car, please e-mail.'

In [31]:
len(dataset.data)

11314

In [28]:
dataset.target_names

['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

In [29]:
dataset.target

array([7, 4, 4, ..., 3, 1, 8])

In [30]:
dataset.DESCR

'.. _20newsgroups_dataset:\n\nThe 20 newsgroups text dataset\n------------------------------\n\nThe 20 newsgroups dataset comprises around 18000 newsgroups posts on\n20 topics split in two subsets: one for training (or development)\nand the other one for testing (or for performance evaluation). The split\nbetween the train and test set is based upon a messages posted before\nand after a specific date.\n\nThis module contains two loaders. The first one,\n:func:`sklearn.datasets.fetch_20newsgroups`,\nreturns a list of the raw texts that can be fed to text feature\nextractors such as :class:`~sklearn.feature_extraction.text.CountVectorizer`\nwith custom parameters so as to extract feature vectors.\nThe second one, :func:`sklearn.datasets.fetch_20newsgroups_vectorized`,\nreturns ready-to-use features, i.e., it is not necessary to use a feature\nextractor.\n\n**Data Set Characteristics:**\n\n=================   ==========\nClasses                     20\nSamples total            18846\nDime

In [14]:
categories = [
    "rec.sport.baseball",
    "sci.space",
    "comp.graphics",
    "talk.religion.misc",
    "alt.atheism"
]

data = fetch_20newsgroups(
    subset="all",
    categories=categories,
    remove=("headers", "footers", "quotes"),
    shuffle=True,
    random_state=42
)

In [17]:
print(len(data.data))
print(data.target_names)
print(data.data[0][:500])

4381
['alt.atheism', 'comp.graphics', 'rec.sport.baseball', 'sci.space', 'talk.religion.misc']

I haven't seen any speculation about it. But, the Salyut KB (Design Bureau) 
was planning a new LH/LOX second stage for the Proton which would boost
payload to LEO from about 21000 to 31500 kg. (Geostationary goes from
2600 kg. (Gals launcher version) to 6000 kg.. This scheme was competing
with the Energia-M last year and I haven't heard which won, except now
I recently read that the Central Specialized KB was working on the 
successor to the Soyuz booster which must be the Energia-M. So the ea


# ■ LDA 모델

In [32]:
vectorizer = CountVectorizer(
    max_df = 0.95,      # 전체 문서의 95%이상 등장하는 단어 제거
    min_df = 5,         # 5개 미만 문서에만 등장하는 단어 제거
    stop_words = "english",
    max_features = 5000
)

X_counts = vectorizer.fit_transform(data.data)
# 실제 문서를 단어 count matrix로 변환

In [33]:
n_topics = 5

lda = LatentDirichletAllocation(
    n_components = n_topics,
    random_state = 42,
    learning_method = "batch",
    max_iter = 20
)

lda.fit(X_counts)

LatentDirichletAllocation(max_iter=20, n_components=5, random_state=42)

※ LDA는 내부적으로 크게 두 가지를 만듦
- Topic_word distribution : 각 topic이 어떤 단어를 많이 가지는지
    - ` Topic 0` : space, nasa, orbit...
- Document-topic distribution : 각 문서가 어떤 toipic비율로 구성되는지
    - ` Document 0` : Topic 0 = 0.70, Topic 1 = 0.10 ...

## ■ Topic별 대표 단어 출력

In [34]:
feature_names = vectorizer.get_feature_names_out()

def display_topics(model, feature_names, n_top_words = 10):
    for topic_idx, topic in enumerate(model.components_):
            # topic은 하나의 topic에 대한 단어 weight 배열
        top_word_indices = topic.argsort()[::-1][:n_top_words]
                            # argsort : 정렬된 순서의 index 반환
                            # 다 슬라이스하되, 역으로
        top_words = [feature_names[i] for i in top_word_indices]
        print(f"Topic {topic_idx}:")
        print(", ".join(top_words))
        print()

display_topics(lda, feature_names, n_top_words = 15)

Topic 0:
year, game, team, think, don, good, games, time, hit, just, baseball, like, better, players, runs

Topic 1:
edu, graphics, image, data, ftp, available, mail, software, pub, information, computer, send, thanks, 3d, images

Topic 2:
god, people, don, think, just, know, say, does, like, jesus, believe, way, said, good, point

Topic 3:
space, earth, nasa, launch, orbit, shuttle, time, new, moon, mission, solar, spacecraft, satellite, years, like

Topic 4:
jpeg, image, file, gif, color, images, bit, format, files, 03, 00, 02, use, version, 04



In [35]:
print(categories)

['rec.sport.baseball', 'sci.space', 'comp.graphics', 'talk.religion.misc', 'alt.atheism']


※ 추측
- Topic 0 : baseball
- Topic 1 : graphics?
- Topic 2 : religion / atheism
- Topic 3 : space
- Topic 4 : graphics?

##■ 각 문서별 토픽비율 확인

In [37]:
doc_topic_dist = lda.transform(X_counts)
doc_topic_df = pd.DataFrame(
    doc_topic_dist,
    columns = [f"Topic {i}" for i in range(n_topics)]
)

doc_topic_df.head()

,Topic 0,Topic 1,Topic 2,Topic 3,Topic 4
0,0.001790,0.001790,0.164333,0.830284,0.001803
1,0.075748,0.006218,0.006268,0.696382,0.215384
2,0.004802,0.004834,0.771253,0.004808,0.214303
3,0.104727,0.002079,0.357448,0.533645,0.002102
4,0.011466,0.011309,0.954742,0.011227,0.011257


In [46]:
topic_cols = [f"Topic {i}" for i in range (n_topics)]

doc_topic_df["dominant_topic"] = doc_topic_df[topic_cols].idxmax(axis=1)
doc_topic_df["dominant_topic_score"] = doc_topic_df[topic_cols].max(axis=1)

doc_topic_df.sort_values("dominant_topic_score", ascending = False)

,Topic 0,Topic 1,Topic 2,Topic 3,Topic 4,dominant_topic,dominant_topic_score
3389,0.000041,0.999835,0.000041,0.000041,0.000042,Topic 1,0.999835
4320,0.000041,0.999833,0.000041,0.000042,0.000042,Topic 1,0.999833
835,0.000044,0.000044,0.000044,0.000044,0.999824,Topic 4,0.999824
3119,0.000044,0.000044,0.000044,0.000044,0.999823,Topic 4,0.999823
3647,0.000044,0.000045,0.000045,0.000044,0.999822,Topic 4,0.999822
...,...,...,...,...,...,...,...
1065,0.200000,0.200000,0.200000,0.200000,0.200000,Topic 0,0.200000
1114,0.200000,0.200000,0.200000,0.200000,0.200000,Topic 0,0.200000
3153,0.200000,0.200000,0.200000,0.200000,0.200000,Topic 0,0.200000
40,0.200000,0.200000,0.200000,0.200000,0.200000,Topic 0,0.200000


## ■ 실제 텍스트 확인

In [48]:
documents = pd.DataFrame({
    "text" : data.data,
    "true_label" : [data.target_names[i] for i in data.target]
})

result_df = pd.concat([documents, doc_topic_df], axis=1)
result_df.head()

,text,true_label,Topic 0,Topic 1,Topic 2,Topic 3,Topic 4,dominant_topic,dominant_topic_score
0,\nI haven't seen any speculation about it. But...,sci.space,0.001790,0.001790,0.164333,0.830284,0.001803,Topic 3,0.830284
1,"\nI'm not sure if this is a big issue, but it ...",sci.space,0.075748,0.006218,0.006268,0.696382,0.215384,Topic 3,0.696382
2,": In article <11838@vice.ICO.TEK.COM>, bobbe@v...",alt.atheism,0.004802,0.004834,0.771253,0.004808,0.214303,Topic 2,0.771253
3,"Oddly, enough, The smithsonian calls the lind...",sci.space,0.104727,0.002079,0.357448,0.533645,0.002102,Topic 3,0.533645
4,"\nGood point. If you haven't read ""The True B...",talk.religion.misc,0.011466,0.011309,0.954742,0.011227,0.011257,Topic 2,0.954742


In [49]:
def show_representative_docs(df, topic_name, n_docs=3, text_length=500):
    topic_docs = df.sort_values(by=topic_name, ascending=False).head(n_docs)

    for idx, row in topic_docs.iterrows():
        print("=" * 80)
        print(f"Document index : {idx}")
        print(f"{topic_name} score : {row[topic_name]:.3f}")
        print("-" * 80)
        print(row["text"][:text_length])
        print()

In [51]:
show_representative_docs(result_df, "Topic 2")
# religion 보다는 antheism

Document index : 4164
Topic 2 score : 1.000
--------------------------------------------------------------------------------
Archive-name: atheism/introduction
Alt-atheism-archive-name: introduction
Last-modified: 5 April 1993
Version: 1.2

-----BEGIN PGP SIGNED MESSAGE-----

                          An Introduction to Atheism
                       by mathew <mathew@mantis.co.uk>

This article attempts to provide a general introduction to atheism.  Whilst I
have tried to be as neutral as possible regarding contentious issues, you
should always remember that this document represents only one viewpoint.  I
would encou

Document index : 3962
Topic 2 score : 0.999
--------------------------------------------------------------------------------
Archive-name: atheism/logic
Alt-atheism-archive-name: logic
Last-modified: 5 April 1993
Version: 1.4

                       Constructing a Logical Argument

Although there is much argument on Usenet, the general quality of argument
found is poor. 

In [52]:
show_representative_docs(result_df, "Topic 4")
# graphics

Document index : 835
Topic 4 score : 1.000
--------------------------------------------------------------------------------
Archive-name: jpeg-faq
Last-modified: 16 May 1993

This FAQ article discusses JPEG image compression.  Suggestions for
additions and clarifications are welcome.

New since version of 2 May 1993:
  * Added info on ImageViewer for NeXT.


This article includes the following sections:

[1]  What is JPEG?
[2]  Why use JPEG?
[3]  When should I use JPEG, and when should I stick with GIF?
[4]  How well does JPEG compress images?
[5]  What are good "quality" settings for JPEG?
[6]  Where can I get JPEG 

Document index : 3119
Topic 4 score : 1.000
--------------------------------------------------------------------------------
Archive-name: jpeg-faq
Last-modified: 2 May 1993

This FAQ article discusses JPEG image compression.  Suggestions for
additions and clarifications are welcome.

New since version of 18 April 1993:
  * New version of XV supports 24-bit viewing for X 

In [53]:
show_representative_docs(result_df, "Topic 1")
# graphics

Document index : 3389
Topic 1 score : 1.000
--------------------------------------------------------------------------------
Archive-name: graphics/resources-list/part1
Last-modified: 1993/04/27


Computer Graphics Resource Listing : WEEKLY POSTING [ PART 1/3 ]
Last Change : 27 April 1993

Many FAQs, including this Listing, are available on the archive site
pit-manager.mit.edu (alias rtfm.mit.edu) [18.172.1.27] in the directory
pub/usenet/news.answers.  The name under which a FAQ is archived appears
in the Archive-name line at the top of the article.
This FAQ is arch

Document index : 4320
Topic 1 score : 1.000
--------------------------------------------------------------------------------
Archive-name: graphics/resources-list/part1
Last-modified: 1993/04/17


Computer Graphics Resource Listing : WEEKLY POSTING [ PART 1/3 ]
Last Change : 17 April 1993

Many FAQs, including this Listing, are available on the archive site
pit-manager.mit.edu (alias rtfm.mit.edu) [18.172.1.27] in the dir

In [54]:
topic_word_table = []

for topic_idx, topic in enumerate(lda.components_):
    top_indices = topic.argsort()[::-1][:15]
    top_words = [feature_names[i] for i in top_indices]
    topic_word_table.append({
        "topic" : f"Topic {topic_idx}",
        "top_words" : ", ".join(top_words)
    })

topic_word_df = pd.DataFrame(topic_word_table)
topic_word_df

,topic,top_words
0,Topic 0,"year, game, team, think, don, good, games, tim..."
1,Topic 1,"edu, graphics, image, data, ftp, available, ma..."
2,Topic 2,"god, people, don, think, just, know, say, does..."
3,Topic 3,"space, earth, nasa, launch, orbit, shuttle, ti..."
4,Topic 4,"jpeg, image, file, gif, color, images, bit, fo..."


※ 결과 분석
- Topic 1, Topic 4 모두 graphics
- Topic 2는 religion / atheism을 모두 포함
- religion / atheism 을 구분하지 못하고, graphics를 두 개로 나눈 것으로 보임(확장자명이 강하게 나오는것과, 일반적인 그래픽 관련 단어를 분리한 듯)

# ■ bigram 실험

In [55]:
vectorizer_bi = CountVectorizer(
    max_df = 0.95,
    min_df = 5,
    ngram_range = (1,2),
    stop_words = "english",
    max_features = 5000
)

X_counts_bi = vectorizer_bi.fit_transform(data.data)
# 실제 문서를 단어 count matrix로 변환

In [57]:
n_topics = 5

lda_bi = LatentDirichletAllocation(
    n_components = n_topics,
    random_state = 42,
    learning_method = "batch",
    max_iter = 20
)

lda_bi.fit(X_counts_bi)

LatentDirichletAllocation(max_iter=20, n_components=5, random_state=42)

In [58]:
feature_names_bi = vectorizer_bi.get_feature_names_out()

display_topics(lda_bi, feature_names_bi, n_top_words = 15)

Topic 0:
god, don, does, just, think, people, say, believe, religion, way, like, point, know, evidence, true

Topic 1:
image, edu, graphics, jpeg, file, images, software, available, ftp, data, files, gif, format, use, mail

Topic 2:
year, game, don, think, just, good, team, time, like, games, hit, better, baseball, know, players

Topic 3:
people, god, jesus, don, think, know, like, just, right, said, did, say, life, good, make

Topic 4:
space, nasa, earth, launch, orbit, shuttle, new, moon, mission, solar, 00, satellite, time, spacecraft, 03



※ 추정
- Topic 0 : religion
- Topic 1 : graphics
- Topic 2 : baseball
- Topic 3 : atheism
- Topic 4 : space

In [61]:
doc_topic_dist_bi = lda_bi.transform(X_counts_bi)
doc_topic_df_bi = pd.DataFrame(
    doc_topic_dist_bi,
    columns = [f"Topic {i}" for i in range(n_topics)]
)

doc_topic_df_bi.head()

,Topic 0,Topic 1,Topic 2,Topic 3,Topic 4
0,0.198399,0.001783,0.107751,0.001791,0.690277
1,0.006607,0.270052,0.006652,0.413449,0.303240
2,0.527206,0.004198,0.423488,0.040916,0.004192
3,0.002110,0.002100,0.136257,0.453678,0.405855
4,0.959490,0.010118,0.010222,0.010121,0.010050


In [62]:
topic_cols = [f"Topic {i}" for i in range (n_topics)]

doc_topic_df_bi["dominant_topic"] = doc_topic_df_bi[topic_cols].idxmax(axis=1)
doc_topic_df_bi["dominant_topic_score"] = doc_topic_df_bi[topic_cols].max(axis=1)

doc_topic_df_bi.sort_values("dominant_topic_score", ascending = False)

,Topic 0,Topic 1,Topic 2,Topic 3,Topic 4,dominant_topic,dominant_topic_score
3389,0.000038,0.999846,0.000039,0.000038,0.000039,Topic 1,0.999846
4320,0.000039,0.999845,0.000039,0.000039,0.000039,Topic 1,0.999845
835,0.000041,0.999838,0.000041,0.000041,0.000040,Topic 1,0.999838
3119,0.000041,0.999837,0.000041,0.000041,0.000040,Topic 1,0.999837
3647,0.000041,0.999836,0.000041,0.000041,0.000041,Topic 1,0.999836
...,...,...,...,...,...,...,...
1806,0.200000,0.200000,0.200000,0.200000,0.200000,Topic 0,0.200000
1160,0.200000,0.200000,0.200000,0.200000,0.200000,Topic 0,0.200000
1813,0.200000,0.200000,0.200000,0.200000,0.200000,Topic 0,0.200000
2550,0.200000,0.200000,0.200000,0.200000,0.200000,Topic 0,0.200000


In [63]:
result_df_bi = pd.concat([documents, doc_topic_df_bi], axis=1)
result_df_bi.head()

,text,true_label,Topic 0,Topic 1,Topic 2,Topic 3,Topic 4,dominant_topic,dominant_topic_score
0,\nI haven't seen any speculation about it. But...,sci.space,0.198399,0.001783,0.107751,0.001791,0.690277,Topic 4,0.690277
1,"\nI'm not sure if this is a big issue, but it ...",sci.space,0.006607,0.270052,0.006652,0.413449,0.303240,Topic 3,0.413449
2,": In article <11838@vice.ICO.TEK.COM>, bobbe@v...",alt.atheism,0.527206,0.004198,0.423488,0.040916,0.004192,Topic 0,0.527206
3,"Oddly, enough, The smithsonian calls the lind...",sci.space,0.002110,0.002100,0.136257,0.453678,0.405855,Topic 3,0.453678
4,"\nGood point. If you haven't read ""The True B...",talk.religion.misc,0.959490,0.010118,0.010222,0.010121,0.010050,Topic 0,0.959490


In [70]:
for i in range(n_topics):
    print(show_representative_docs(result_df_bi, f"Topic {i}", 2))

Document index : 3962
Topic 0 score : 0.999
--------------------------------------------------------------------------------
Archive-name: atheism/logic
Alt-atheism-archive-name: logic
Last-modified: 5 April 1993
Version: 1.4

                       Constructing a Logical Argument

Although there is much argument on Usenet, the general quality of argument
found is poor.  This article attempts to provide a gentle introduction to
logic, in the hope of improving the general level of debate.

Logic is the science of reasoning, proof, thinking, or inference [Concise
OED].  Logic allows us to analyze a piece of reasoning an

Document index : 748
Topic 0 score : 0.999
--------------------------------------------------------------------------------
#In article <1r3tqo$ook@horus.ap.mchp.sni.de>
# 
#>#>|>#>#Theism is strongly correlated with irrational belief in absolutes. Irrational
#>#>|>#>#belief in absolutes is strongly correlated with fanatism.
#>#
#>#(deletion)
#>#
#>#>|Theism is correlate

※ 결과 분석
- Topic 0 : atheism
- Topic 1 : graphics
- Topic 2 : baseball
- Topic 3 : religion
- Topic 4 : space

- unigram보다 확연히 잘 구분하고 있음

In [83]:
topic_label_map = {
    "Topic 0": "alt.atheism",
    "Topic 1": "comp.graphics",
    "Topic 2": "rec.sport.baseball",
    "Topic 3": "talk.religion.misc",
    "Topic 4": "sci.space"
}

In [84]:
result_df_bi["predicted_label"] = (
    result_df_bi["dominant_topic"].map(topic_label_map)
)

result_df_bi.head()

,text,true_label,Topic 0,Topic 1,Topic 2,Topic 3,Topic 4,dominant_topic,dominant_topic_score,predicted_label,correct
0,\nI haven't seen any speculation about it. But...,sci.space,0.198399,0.001783,0.107751,0.001791,0.690277,Topic 4,0.690277,sci.space,True
1,"\nI'm not sure if this is a big issue, but it ...",sci.space,0.006607,0.270052,0.006652,0.413449,0.303240,Topic 3,0.413449,talk.religion.misc,False
2,": In article <11838@vice.ICO.TEK.COM>, bobbe@v...",alt.atheism,0.527206,0.004198,0.423488,0.040916,0.004192,Topic 0,0.527206,alt.atheism,False
3,"Oddly, enough, The smithsonian calls the lind...",sci.space,0.002110,0.002100,0.136257,0.453678,0.405855,Topic 3,0.453678,talk.religion.misc,False
4,"\nGood point. If you haven't read ""The True B...",talk.religion.misc,0.959490,0.010118,0.010222,0.010121,0.010050,Topic 0,0.959490,alt.atheism,False


In [86]:
result_df_bi["correct"] = ((result_df_bi["predicted_label"]) == (result_df_bi["true_label"]))

result_df_bi["correct"].value_counts()

# 완벽히 구분되지는 않지만, 약 6-70%는 맞춤

,count
correct,
True,3030
False,1351


In [81]:
pd.crosstab(
    result_df_bi["true_label"],
    result_df_bi["dominant_topic"]
)

dominant_topic,Topic 0,Topic 1,Topic 2,Topic 3,Topic 4
true_label,,,,,
alt.atheism,483,14,54,230,18
comp.graphics,91,806,37,13,26
rec.sport.baseball,84,29,852,17,12
sci.space,121,89,73,108,596
talk.religion.misc,265,9,39,293,22


In [82]:
pd.crosstab(
    result_df["true_label"],
    result_df["dominant_topic"]
)
# 명확하게 1대1 대응되는 것이 없음 확인 가능
# atheism, religion, space가 topic2로 잡힌 것이 특징적임

dominant_topic,Topic 0,Topic 1,Topic 2,Topic 3,Topic 4
true_label,,,,,
alt.atheism,39,10,711,20,19
comp.graphics,40,612,98,12,211
rec.sport.baseball,868,37,75,4,10
sci.space,71,92,205,612,7
talk.religion.misc,43,9,556,17,3


# ■ Conclusion

This project explored unsupervised topic modeling using LDA on the 20 Newsgroups dataset.

The unigram-based model produced several mixed topics, particularly combining religion and atheism-related documents into a shared topic due to strong vocabulary overlap. Some categories, such as computer graphics, were also split into multiple subtopics reflecting different discussion patterns.

After introducing bigrams with `ngram_range=(1,2)`, topic separation became noticeably clearer. Phrase-level features such as "space shuttle" and "file format" provided more specific semantic signals than individual words alone, resulting in better alignment between dominant topics and the original categories.

The experiment showed that topic modeling results are highly sensitive to text representation and preprocessing choices. It also demonstrated an important difference between supervised classification and unsupervised NLP: topic models discover statistical structures in text rather than directly predicting predefined labels.

Overall, this project provided a practical introduction to unsupervised NLP, topic interpretation, and representation-based analysis.